# Docs 3 — Cleaning at Scale

Small functions, one fix each, composed in order, every change logged. The
log is your alibi.

In [ ]:
# The pile: six monthly reports from a community food pantry.
# Three are clean, three carry the classic damage (headers, spacing, OCR).
pile = {
 "jan.txt": """Date: 2026-01-16
Families served: 167
Donations received: $1,210.00
Contact: pantry@example.org""",
 "feb.txt": """Date: 2026-02-13
Families served: 174
Donations received: $1,385.50
Contact: pantry@example.org""",
 "mar_extracted.txt": """COMMUNITY FOOD PANTRY \u2014 MONTHLY REPORT
Page 1 of 1   PANTRY-MAR-FINAL
Date:  March 14, 2026
Families served:212
Donations received : $1,847.50
Contact: pantry@example.org""",
 "apr_extracted.txt": """Page 1 of 1   PANTRY-APR-FINAL
Date: 4/11/26
Families   served: 198
Donations received: $2,210.00
contact: pantry@example.org""",
 "may_ocr.txt": """Date: May 9, 2026
Families served: 241
Donations received: $l,655.25
Contact: pantry@example.org""",
 "jun_note.txt": """Quick note instead of the form this month, sorry! We had a
great June \u2014 somewhere around fifteen hundred dollars came in between the
two drives, and I counted 188 families across the month. \u2014 Rosa""",
}
for name, text in pile.items():
    print(f"--- {name} ({len(text)} chars)")
    print(text[:120].replace(chr(10), " / "))

In [ ]:
import re

LOG = []

def strip_headers(text):
    before = len(text.splitlines())
    text = "\n".join(l for l in text.splitlines() if not re.match(r"^Page \d+ of \d+", l))
    removed = before - len(text.splitlines())
    if removed: LOG.append(f"strip_headers: removed {removed} header line(s)")
    return text

def fix_spacing(text):
    runs = len(re.findall(r"  +", text))
    text = re.sub(r"  +", " ", text)
    joined = len(re.findall(r"(?<=[a-z]):(?=\d)", text))
    text = re.sub(r"([a-z]):(\d)", r"\1: \2", text)
    if runs or joined: LOG.append(f"fix_spacing: {runs} whitespace runs, {joined} joined fields")
    return text

def fix_punct(text):
    n = len(re.findall(r" :", text))
    text = text.replace(" :", ":")
    if n: LOG.append(f"fix_punct: {n} floating colon(s)")
    return text

def standardize_dates(text):
    n = 0
    months = {"January":1,"February":2,"March":3,"April":4,"May":5,"June":6,
              "July":7,"August":8,"September":9,"October":10,"November":11,"December":12}
    def name_style(m):
        nonlocal n; n += 1
        return f"{m.group(3)}-{months[m.group(1)]:02d}-{int(m.group(2)):02d}"
    text = re.sub(r"(" + "|".join(months) + r") (\d{1,2}), (\d{4})", name_style, text)
    def slash_style(m):
        nonlocal n; n += 1
        y = m.group(3) if len(m.group(3)) == 4 else "20" + m.group(3)
        return f"{y}-{int(m.group(1)):02d}-{int(m.group(2)):02d}"
    text = re.sub(r"\b(\d{1,2})/(\d{1,2})/(\d{2,4})\b", slash_style, text)
    if n: LOG.append(f"standardize_dates: rewrote {n} date(s)")
    return text

def clean(text):
    # Order matters: headers out first, then spacing, then punctuation, then dates.
    for step in (strip_headers, fix_spacing, fix_punct, standardize_dates):
        text = step(text)
    return text

In [ ]:
cleaned = {}
for name, raw in pile.items():
    LOG.append(f"--- {name}")
    cleaned[name] = clean(raw)

print("THE LOG:")
for line in LOG:
    print(" ", line)
print()
print("SAMPLE (mar_extracted.txt, after cleaning):")
print(cleaned["mar_extracted.txt"])

Read the log top to bottom: it answers "what did processing change?"
with a record instead of a shrug.

## The mystery document

This one has junk none of the four functions handle. Diagnose it, then
write your fifth cleaner.

In [ ]:
mystery = """Date: 2026-07-11\u00a0
Families\u00a0served: 176
Donations received: $980.00\u00a0\u00a0
Contact: pantry@example.org"""

print(repr(mystery[:30]))
# Hint: repr() shows characters that print() hides. What is \u00a0,
# and which of your patterns does it silently break?

In [ ]:
def your_cleaner(text):
    # TODO: one fix, logged. Where does it belong in clean()'s order, and why?
    return text

## Turn-in

Your fifth cleaner, one sentence on its position in the order, and the log
line it produced when you reran the pile.